## 필수과제1 (타이타닉데이터셋)
- VarianceThreshold -타이타닉 데이터 feature_selection
    - 임계값 기준을 몇으로 했는지?
    - 그 기준의 이유
    - 어떤 식으로 찾았는지!
- 어떤 피처가 선택이 되었나?

In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, f1_score

In [84]:
# 데이터 로드
tt = sns.load_dataset('titanic')
tt

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [85]:
tt.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [86]:
# 결측치 처리
tt['age'].fillna(tt['age'].median(), inplace = True)
tt['deck'].fillna(tt['deck'].mode()[0], inplace = True)
tt['embarked'].fillna(tt['embarked'].mode()[0], inplace = True)
tt['embark_town'].fillna(tt['embark_town'].mode()[0], inplace = True)

In [87]:
tt.isna().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
dtype: int64

In [88]:
# 사용할 피처를 정리
X = tt[['pclass','sex','age','fare','embark_town']]
y = tt['survived']

In [89]:
# 연속형 변수를 범주형
# 사분위수로 

X.loc[:, 'age_binned']=pd.qcut(X['age'], q=4, labels=False)
X.loc[:, 'fare_binned']=pd.qcut(X['fare'], q=4, labels=False)

C:\Users\ghfjg\AppData\Local\Temp\ipykernel_7440\2055354963.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.loc[:, 'age_binned']=pd.qcut(X['age'], q=4, labels=False)
C:\Users\ghfjg\AppData\Local\Temp\ipykernel_7440\2055354963.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.loc[:, 'fare_binned']=pd.qcut(X['fare'], q=4, labels=False)


In [90]:
# 원핫 인코더로 잡아서 진행
from sklearn.preprocessing import OneHotEncoder
X=X.drop(['age','fare'], axis=1)
onehot_encoder =OneHotEncoder(sparse_output=False, drop='first')

In [91]:
X_encoded = onehot_encoder.fit_transform(X)

In [92]:
X_encoded

array([[0., 1., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 1., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 1., 0.],
       [0., 1., 1., ..., 0., 0., 0.]])

In [100]:
# 분산의 임계값 계산
selector = VarianceThreshold(threshold = 0.2)
X_high_variance =selector.fit_transform(X_encoded)
X_high_variance

array([[1., 1., 0.],
       [0., 0., 0.],
       [1., 0., 1.],
       ...,
       [1., 0., 1.],
       [0., 1., 1.],
       [1., 1., 0.]])

In [101]:
# 분산 값과 선택된 특성 이름 가져오기
variances = selector.variances_
features = onehot_encoder.get_feature_names_out(X.columns)

In [102]:
# 임계값에 따른 선택된 특성 필터링
selected_features = features[selector.get_support()]

In [103]:
# 특성과 분산 값을 포함한 DataFrame 생성
variance_df = pd.DataFrame({
    'Feature': features,
    'Variance': variances
}).sort_values(by='Variance', ascending=True)
variance_df 

,Feature,Variance
3,embark_town_Queenstown,0.078951
6,age_binned_2,0.128558
0,pclass_2,0.163863
7,age_binned_3,0.184232
9,fare_binned_2,0.187078
10,fare_binned_3,0.187078
8,fare_binned_1,0.188199
4,embark_town_Southampton,0.199362
5,age_binned_1,0.226185
2,sex_male,0.228218


In [105]:
# 선택된 특성만 필터링해서 보여주자
selected_variance_df = variance_df[variance_df['Feature'].isin(selected_features)]
selected_variance_df

,Feature,Variance
5,age_binned_1,0.226185
2,sex_male,0.228218
1,pclass_3,0.247392
